# Silver Clientes

**Fuente:** `brz_dataco.sharepoint.raw_supply_chain` (histórico completo acumulado)

**Destino:** `slv_dataco.supply_chain.md_clientes`

**Grano:** un registro por `customer_id`

## Qué hace

1. Lee todo el histórico de Bronze acumulado, no solo el último corte.
2. Limpia y normaliza los datos:

   * Campos de texto: `trim` + mayúsculas.
   * Identificadores numéricos: `cast` a `int`.
3. Deduplica por `customer_id`, priorizando el registro del corte más reciente (`dedup_reciente`, `fecha_fin_corte` descendente).
4. Marca la calidad de cada registro (ver más abajo).
5. Conserva las columnas de trazabilidad provenientes de Bronze.
6. Escribe la tabla con `MERGE` por `customer_id`, como el procedimiento original (`usp_tskInsDataMart_Clientes`).
7. Registra en el esquema de control cuántas filas se agruparon y cuántas quedaron en cuarentena.

## Calidad y cuarentena

Ningún registro se elimina. Los que incumplen una regla se conservan marcados y Gold no los lee.

| Columna | Valores |
|---|---|
| `estado_calidad` | `VALIDO` o `CUARENTENA` |
| `motivo_cuarentena` | `CLIENTE_ID_INVALIDO` o `NULL` |
| `fecha_procesamiento` | Momento en que Silver procesó el registro |

## Trazabilidad conservada

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`

## Auditoría de la ejecución

Al terminar, el notebook deja evidencia en `metadata_dataco.supply_chain`:

* `hd_deduplicacion_log`: una fila por ejecución, con los clientes únicos, las filas agrupadas (alto y esperable, porque un cliente se repite en cada ítem) y los duplicados reales por `order_id` + `order_item_id`, que deben ser **0**.
* `hd_cuarentena_resumen`: filas aisladas por regla en esa ejecución, incluso cuando son 0.

In [0]:
# ============================================================
# INICIO DE LOG DE EJECUCIÓN — silver_clientes
# ============================================================

import uuid

dbutils.widgets.text("metadata_catalog", "metadata_dataco", "Catálogo Metadata")
dbutils.widgets.text("metadata_schema", "supply_chain", "Esquema Metadata")
dbutils.widgets.text("id_ejecucion_manual", "", "Id de ejecución (solo corridas manuales)")

metadata_catalog = dbutils.widgets.get("metadata_catalog")
metadata_schema = dbutils.widgets.get("metadata_schema")

TABLA_ETL_LOG = f"{metadata_catalog}.{metadata_schema}.hd_etl_log"
TABLA_DEDUP_LOG = f"{metadata_catalog}.{metadata_schema}.hd_deduplicacion_log"
TABLA_CUARENTENA_RESUMEN = f"{metadata_catalog}.{metadata_schema}.hd_cuarentena_resumen"

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=dbutils.widgets.get("id_ejecucion_manual")
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "slv_customer"
CAPA = "silver"

spark.sql(f"""
INSERT INTO {TABLA_ETL_LOG}
(
    log_id,
    id_ejecucion,
    notebook,
    capa,
    estado,
    fecha_inicio
)
VALUES
(
    '{LOG_ID}',
    '{id_ejecucion}',
    '{NOTEBOOK}',
    '{CAPA}',
    'EN_PROCESO',
    current_timestamp()
)
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")
print("✓ log_id publicado como Task Value")

In [0]:
# ============================================================
# Celda 1: Parámetros + lectura de Bronze
# ------------------------------------------------------------
# Define los widgets del notebook (catálogo, esquema y tabla
# Bronze/Silver) y lee TODO el histórico acumulado en Bronze
# (no solo el último corte), porque silver_clientes se
# reconstruye completo cada corrida (mismo criterio que el
# TRUNCATE+dedup del SP original).
# ============================================================

dbutils.widgets.text("bronze_catalog", "brz_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "sharepoint", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_supply_chain", "Tabla Bronze")

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table = dbutils.widgets.get("bronze_table")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

BRONZE_TABLE_FQN = f"{bronze_catalog}.{bronze_schema}.{bronze_table}"
SILVER_TABLE_FQN = f"{silver_catalog}.{silver_schema}.md_clientes"

df_bronze = spark.table(BRONZE_TABLE_FQN)

print(f"Total filas en Bronze: {df_bronze.count()}")
print(f"Origen: {BRONZE_TABLE_FQN}")
print(f"Destino: {SILVER_TABLE_FQN}")

In [0]:
# Celda 2: Helpers de limpieza
# ------------------------------------------------------------
# clean_str  -> equivalente a UPPER(TRIM(ISNULL(col, '')))
# clean_num  -> equivalente a CAST(ISNULL(NULLIF(TRIM(col),''),'0') AS tipo)
# ============================================================

from pyspark.sql import functions as F, Window

def clean_str(col_name: str, default: str = "") -> F.Column:
    return F.upper(F.trim(F.coalesce(F.col(col_name).cast("string"), F.lit(default))))

def clean_num(col_name: str, cast_type, default: str = "0") -> F.Column:
    trimmed = F.trim(F.col(col_name).cast("string"))
    nulled = F.when((trimmed.isNull()) | (trimmed == ""), None).otherwise(trimmed)
    return F.coalesce(nulled, F.lit(default)).cast(cast_type)

print("✓ Helpers definidos: clean_str, clean_num")

In [0]:
[c for c in df_bronze.columns if "customer" in c.lower()]

In [0]:
# ============================================================
# Celda 3: Construcción de md_clientes + deduplicación determinista
# ------------------------------------------------------------
# 1) Limpia y tipifica columnas desde Bronze (sin deduplicar aún)
# 2) Deduplica por customer_id, priorizando el corte más reciente
#    (fecha_fin_corte descendente) — ya NO usa el criterio arbitrario
#    del SP original (ORDER BY (SELECT NULL))
# 3) Conserva las columnas de trazabilidad provenientes de Bronze
# ============================================================

df_silver_clientes = df_bronze.select(
    clean_num("customer_id", "int").alias("customer_id"),
    clean_str("customer_fname").alias("customer_fname"),
    clean_str("customer_lname").alias("customer_lname"),
    clean_str("customer_email").alias("customer_email"),
    clean_str("customer_segment").alias("customer_segment"),
    clean_str("customer_city").alias("customer_city"),
    clean_str("customer_state").alias("customer_state"),
    clean_str("customer_street").alias("customer_street"),
    clean_str("customer_zipcode").alias("customer_zipcode"),
    clean_str("customer_country").alias("customer_country"),

    # --------------------------------------------------------
    # Trazabilidad
    # --------------------------------------------------------
    F.col("fecha_inicio_corte"),
    F.col("fecha_fin_corte"),
    F.col("fecha_carga"),
    F.col("id_ejecucion")
)

# ------------------------------------------------------------
# Deduplicación determinista — prioriza el corte más reciente
# ------------------------------------------------------------
from pyspark.sql import Window

def dedup_reciente(df, partition_cols: list, orden_col: str):
    w = Window.partitionBy(*partition_cols).orderBy(F.col(orden_col).desc())
    return (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

# Cuarentena: customer_id nulo, vacío o 0 -> CLIENTE_ID_INVALIDO.
# Sin clave no hay con qué identificar al cliente, así que estas filas se
# agrupan en un solo registro (customer_id = 0) marcado en cuarentena.
df_silver_clientes = (
    dedup_reciente(df_silver_clientes, partition_cols=["customer_id"], orden_col="fecha_fin_corte")
    .withColumn("motivo_cuarentena",
                F.when(F.col("customer_id") == 0, F.lit("CLIENTE_ID_INVALIDO")))
    .withColumn("estado_calidad",
                F.when(F.col("motivo_cuarentena").isNotNull(), F.lit("CUARENTENA"))
                 .otherwise(F.lit("VALIDO")))
    .withColumn("fecha_procesamiento", F.current_timestamp())
)

print(f"Filas en Bronze (antes de dedup) : {df_bronze.count()}")
print(f"Filas en Silver (después del dedup determinista) : {df_silver_clientes.count()}")

display(df_silver_clientes.limit(10))

In [0]:
# ============================================================
# Celda 3b: Crear esquema Silver (el catálogo slv_dataco ya existe)
# ============================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{silver_catalog}.{silver_schema}"
)

print(f"✓ Esquema {silver_catalog}.{silver_schema} listo")

In [0]:
TABLA_SILVER = SILVER_TABLE_FQN

In [0]:
# ============================================================
# MERGE EN SILVER
# Replica el comportamiento de usp_tskInsDataMart_Clientes
# ============================================================

from delta.tables import DeltaTable

if not spark.catalog.tableExists(TABLA_SILVER):
    (
        df_silver_clientes.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TABLA_SILVER)
    )

    print(f"✓ Tabla {TABLA_SILVER} creada inicialmente")

else:
    delta_target = DeltaTable.forName(spark, TABLA_SILVER)

    (
        delta_target.alias("t")
        .merge(
            df_silver_clientes.alias("s"),
            "t.customer_id = s.customer_id"
        )
        .whenMatchedUpdate(set={
            "customer_fname": "s.customer_fname",
            "customer_lname": "s.customer_lname",
            "customer_email": "s.customer_email",
            "customer_segment": "s.customer_segment",
            "customer_city": "s.customer_city",
            "customer_state": "s.customer_state",
            "customer_street": "s.customer_street",
            "customer_zipcode": "s.customer_zipcode",
            "customer_country": "s.customer_country",
            "estado_calidad": "s.estado_calidad",
            "motivo_cuarentena": "s.motivo_cuarentena",
            "fecha_procesamiento": "s.fecha_procesamiento",
            "fecha_inicio_corte": "s.fecha_inicio_corte",
            "fecha_fin_corte": "s.fecha_fin_corte",
            "fecha_carga": "s.fecha_carga",
            "id_ejecucion": "s.id_ejecucion"
        })
        .whenNotMatchedInsert(values={
            "customer_id": "s.customer_id",
            "customer_fname": "s.customer_fname",
            "customer_lname": "s.customer_lname",
            "customer_email": "s.customer_email",
            "customer_segment": "s.customer_segment",
            "customer_city": "s.customer_city",
            "customer_state": "s.customer_state",
            "customer_street": "s.customer_street",
            "customer_zipcode": "s.customer_zipcode",
            "customer_country": "s.customer_country",
            "estado_calidad": "s.estado_calidad",
            "motivo_cuarentena": "s.motivo_cuarentena",
            "fecha_procesamiento": "s.fecha_procesamiento",
            "fecha_inicio_corte": "s.fecha_inicio_corte",
            "fecha_fin_corte": "s.fecha_fin_corte",
            "fecha_carga": "s.fecha_carga",
            "id_ejecucion": "s.id_ejecucion"
        })
        .execute()
    )

    print(f"✓ MERGE ejecutado sobre {TABLA_SILVER}")

In [0]:
# ============================================================
# Registro de deduplicación y cuarentena de esta ejecución
# ------------------------------------------------------------
# Una fila por ejecución y tabla Silver.
# filas_agrupadas  -> filas de Bronze agrupadas en un registro por cliente
#                     (informativa, alta y esperable: un cliente repite en cada ítem).
# duplicados_reales -> filas repetidas de (order_id, order_item_id) en Bronze; debe ser 0.
# El DELETE previo evita duplicar el registro si la task se reintenta.
# ============================================================
TABLA_LOG = "md_clientes"
MOTIVOS = ["CLIENTE_ID_INVALIDO"]

filas_entrada = df_bronze.count()
filas_salida = df_silver_clientes.count()
filas_unicas_bronze = df_bronze.select("order_id", "order_item_id").distinct().count()
duplicados_reales = filas_entrada - filas_unicas_bronze

spark.sql(
    f"DELETE FROM {TABLA_DEDUP_LOG} WHERE id_ejecucion = :id AND tabla_silver = :tabla",
    args={"id": id_ejecucion, "tabla": TABLA_LOG}
)
spark.sql(
    f"INSERT INTO {TABLA_DEDUP_LOG} VALUES (:id, :tabla, :clave, :entrada, :salida, "
    f":agrupadas, :clave_fisica, :duplicados, current_timestamp())",
    args={"id": id_ejecucion, "tabla": TABLA_LOG, "clave": "customer_id",
          "entrada": filas_entrada, "salida": filas_salida,
          "agrupadas": filas_entrada - filas_salida,
          "clave_fisica": "order_id, order_item_id",
          "duplicados": duplicados_reales}
)

spark.sql(
    f"DELETE FROM {TABLA_CUARENTENA_RESUMEN} WHERE id_ejecucion = :id AND tabla_silver = :tabla",
    args={"id": id_ejecucion, "tabla": TABLA_LOG}
)
df_motivos = spark.createDataFrame([(m,) for m in MOTIVOS], "motivo STRING")
df_conteo = (df_silver_clientes.filter("estado_calidad = 'CUARENTENA'")
             .groupBy("motivo_cuarentena").count())
(
    df_motivos.join(df_conteo, df_motivos.motivo == df_conteo.motivo_cuarentena, "left")
    .select(
        F.lit(id_ejecucion).alias("id_ejecucion"),
        F.lit(TABLA_LOG).alias("tabla_silver"),
        F.col("motivo").alias("motivo_cuarentena"),
        F.coalesce(F.col("count"), F.lit(0)).cast("bigint").alias("filas"),
        F.current_timestamp().alias("fecha_registro"),
    )
    .write.format("delta").mode("append").saveAsTable(TABLA_CUARENTENA_RESUMEN)
)

print(f"✓ {filas_entrada} filas de Bronze -> {filas_salida} clientes | agrupadas: {filas_entrada - filas_salida} | duplicados reales: {duplicados_reales}")

In [0]:
TABLA_DESTINO = SILVER_TABLE_FQN   # cambia por notebook (ver tabla)
filas_procesadas = spark.table(TABLA_DESTINO).count()

spark.sql(f"""
UPDATE {TABLA_ETL_LOG}
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"✓ etl_log actualizado: EXITOSO ({filas_procesadas} filas en {TABLA_DESTINO})")